- todo
    - <|endoftext|> 与 "eos_token": "<|im_end|>", 一个是预训练的，一个是后训练的？
        - https://huggingface.co/XiaomiMiMo/MiMo-V2.5/blob/main/tokenizer_config.json
    - sprinkle chat-template special tokens into the pre-training corpus so P(x|template) is a broad mixture and not a delta on the post-training set.
- vs. Magpie
    - 聊天模板仅仅存在于后训练阶段，用它做条件约束（Conditioning），就会使得模型生成的分布极度逼近它当年做后训练时的真实数据分布 。
    - Magpie 的视角是“数据合成 (Synthesis)”：Magpie 关注的是如何利用这个 trick 来白嫖强大模型的能力，自动化地生成高质量的指令数据集，用于训练其他模型。
    - 本文的视角是“数据泄露与记忆 (Memorization & Extraction)”：本文并没有把这当成一种“生成新数据”的技术，而是将其视为一种安全攻击与数据提取手段。本文指出，模型吐出来的并非全是“新知识”，而是大量原封不动或语义等价的原始私有训练数据 。

- Alignment Data: 包含指令微调 SFT、强化学习 RL 等数据，本文主要关注目前最核心的两种后训练 (Post-training) 技术：
    - SFT (监督微调)：用高质量人工标注的 `Question-Answer` 对直接优化极大似然。
    - RLVR / PPO / GRPO：利用验证奖励机制 (Verifiable Rewards) 对生成内容进行策略梯度更新。例如 Qwen2.5 家族广泛使用的 GRPO 技术。
- 以往的研究在检测模型是否“背诵”了训练数据时，高度依赖字面匹配 (String Matching / Levenshtein Distance)。本文作者发现，这种指标严重低估了记忆现象。如果大模型输出的文本在语义、逻辑、公式结构上与训练集完全一致，仅仅是选项顺序改变或无关词汇替换，字面匹配分数会很低，但它在本质上已经泄露了核心的高质量训练数据。
- 开源模型的一个巨大商业风险：竞争对手完全可以通过简单的提示词攻击，提取出模型的私有对齐数据集，进而直接用于训练自己的模型（即所谓的模型蒸馏 Model Distillation）。这暗示着，模型蒸馏在很多时候实际上就是训练集的数据提取 (Data Extraction)。

### cases

```
user: xxx

<｜begin▁of▁sentence｜>{system}
<｜User｜>xxx
<｜Assistant｜>
```

### Memorization

> 将严格的“字面记忆”拓展到基于高维特征的“近似语义记忆 (Approximate Semantic Memorization)”。
- 模型在训练过程中会记住部分训练数据。之前的研究分为：
    - 精确记忆 (Verbatim Extraction)：一字不差地吐出训练文本。
    - 马赛克记忆 (Mosaic Memory)：模型能将不同片段重新组合。
    - 分布记忆 (Distributional Memorization)：输出分布高度拟合训练集的 n-gram 频率。

### gradient perspective

> Accidental Gradient Alignment)：梯度是如何驱使模型不仅学习答案，甚至“意外”记忆下提示词（Prompt）的。即使在 SFT 中掩码了问题 $Q$ 的梯度，模型仍然会记住 $Q$。

- 假设模型参数为 $\theta$。在针对数据对 $(Q, A)$ 进行训练期间，通过对答案的负对数似然（Negative Log-likelihood）执行梯度下降，参数从 $\theta$ 更新为 $\theta'$：
  $$\theta' = \theta + \eta \nabla_\theta \log P(A|Q; \theta),$$
- 其中 $\eta>0$ 为学习率（Learning Rate）。本推导关注的是在此次更新之后，问题 $Q$ 的对数概率 $\log P(Q; \theta')$ 所发生的变化。对 $\log P(Q; \theta')$ 在 $\theta$ 处进行一阶泰勒展开（First-order Taylor Expansion），可得：
  $$\log P(Q; \theta') \approx \log P(Q; \theta) + (\theta' - \theta)^\top \nabla_\theta \log P(Q; \theta)$$
- 将上述的参数更新规则代入 $(\theta' - \theta)$ 项中，我们获得：
  $$\log P(Q; \theta') \approx \log P(Q; \theta) + \eta (\nabla_\theta \log P(A|Q; \theta))^\top (\nabla_\theta \log P(Q; \theta)).$$
- 若要使问题 $Q$ 的似然概率（Likelihood）有所增加，这两个梯度的内积（Inner Product）必须为正：
  $$(\nabla_\theta \log P(A|Q; \theta))^\top (\nabla_\theta \log P(Q; \theta)) > 0.$$



### experiments

Chat Templates (聊天模板)。特殊 tokens（如 <|user|>、<|im_start|>）通常仅仅是在后训练阶段才引入的。如果用户在开头强行输入这些标志，就会触发模型生成后训练数据的分布。
- SFT 提取实验设置
    - 目标模型: OLMo 2 13B (包含 939k 开源 SFT 数据集)
    - 生成提示词 (Prompt): <|endoftext|><|user|>\n
    - 超参数: Temperature = 1，独立生成 100万 (1M) 条样本。
    - 评估模型: Gemini-embedding-001。将 Q 与 A 拼接，去除特殊 tokens 后进行余弦相似度计算，阈值定为 0.95。对比基线为 Levenshtein/Indel 字面距离 (阈值 0.9)。
    - 蒸馏验证: 过滤出约 930k 合成样本，在 OLMo 2 7B 上使用同样参数重新执行 SFT。
- RL (强化学习) 提取实验设置
    - 目标模型: Open-Reasoner-Zero 7B (基于 Qwen2.5 7B 通过 PPO 训练，包含 57k 数据)
    - 生成提示词 (Prompt): 一个多行的复杂指令模板：
    - 生成数量: 10 万 (100k) 条样本。
    - 蒸馏验证: 使用上述方法生成的 57k 样本，利用 Dr. GRPO 算法在一台未经对齐的 Qwen2.5 7B 底座上重新跑 RL。

**输入:**
* **开放权重模型** $M$
* **后训练特有的聊天模板前缀** $P$ (例如 `<|endoftext|><|user|>\n`)
* **高性能嵌入模型** $E$ (本文使用 gemini-embedding-001)
* **生成数量** $K$ (例如 1M)
* **语义记忆阈值** $\tau$ (设定为 0.95)

**流程:**
1. **Initialize** 集合 $S_{syn} \leftarrow \emptyset$
2. **For** $k = 1$ **to** $K$ **do:**
3. &nbsp;&nbsp;&nbsp;&nbsp;使用前缀 $P$ 提示模型 $M$，采用默认温度 ($T=1$) 进行自由生成
4. &nbsp;&nbsp;&nbsp;&nbsp;获得生成样本 $s_k$，并提取清洗掉特殊符号
5. &nbsp;&nbsp;&nbsp;&nbsp;$S_{syn} \leftarrow S_{syn} \cup \{ s_k \}$
6. **End For**
7. *(验证泄露率阶段)* **For** 每一条真实的训练样本 $d \in D_{train}$ **do:**
8. &nbsp;&nbsp;&nbsp;&nbsp;计算最高相似度: $score = \max_{s \in S_{syn}} [ E(d) \cdot E(s) ]$
9. &nbsp;&nbsp;&nbsp;&nbsp;**If** $score \ge \tau$ **then:**
10. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;判定该样本 $d$ 被模型“近似语义记忆”
11. &nbsp;&nbsp;&nbsp;&nbsp;**End If**
12. **End For**

**输出:**
* 生成的数据集 $S_{syn}$ 即可用于无缝蒸馏新模型。

- 字面匹配(String Matching)严重掩盖了数据泄露
    - 当使用传统的字面匹配（Levenshtein）度量时，记忆率看起来“微乎其微”（分布集中在 0.3-0.6，几乎没有样本超过 0.9 阈值）。但当切换到神经嵌入（Neural Embeddings）时，大量生成的样本被判定为高度记忆（余弦相似度 > 0.95）。
    - 模型往往会对题目中的数字、多选题选项顺序做微小改动，或者改变推理过程的连词，这会毁灭字面匹配分数，但完整的逻辑、模板和语义完全是来自原始数据。
- “用魔法打败魔法”：提取数据能直接还原模型性能
    - 作者提取出 SFT 数据，并重新训练了一个底座模型，在多个标准 Benchmark 上竟然与官方利用内部真实数据集训练的模型表现不相上下！证明了提取出来的数据质量极高。
- 强化学习 (RL) 也会引发硬记忆
    - 相比于 SFT 直接拟合目标概率，RL（如 PPO）仅仅是在最大化期望奖励。直觉上它应该不容易造成对训练集的死记硬背。但结果发现，不仅模型会一字不差地吐出 RL 训练样本（甚至自动补齐了训练集里没有的思维链 ``），而且在 RL 训练后，底座模型对这些 Prompt 的 likelihood (似然值) 出现了从 $10^{-11}$  飙升至 $10^{-5}$ 的恐怖增长。
- gradient unfied perspective
    - 如果在 RL 中，模型针对问题 $Q$ 生成了回答 $A$，并且获得了正向的优势函数（Advantage $\hat{A} > 0$），那么它的策略梯度更新步为 $\hat{A} \cdot \nabla_\theta \log P(A|Q; \theta)$。这在数学形式上，完全等价于以权重 $\hat{A}$ 对这对 $(Q, A)$ 进行了一次 SFT 更新。
    - 只要底层依然是依靠似然度梯度去更新参数，不管外面包装的是 SFT 还是 RL，数据泄露的风险是一视同仁的。